In [ ]:
%pip install -q transformers datasets scikit-learn accelerate evaluate
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00
Looking in indexes: https://download.pytorch.org/whl/cu121


# Version 1

In [ ]:
import os
import json
import torch
import numpy as np
import collections
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
)
from sklearn.metrics import precision_score, recall_score, f1_score

# Check device
assert torch.cuda.is_available(), "CUDA is not available! Install CUDA-enabled PyTorch."
device = torch.device("cuda")
torch.cuda.empty_cache()
print(f"Using device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load the local CUAD JSON file (SQuAD format)
cuad_path = os.path.join("drive/MyDrive/Sem 2/AIG230/Project/Dataset", "CUAD_v1.json")

with open(cuad_path, "r", encoding="utf-8") as f:
    cuad_raw = json.load(f)

print(f"CUAD version: {cuad_raw.get('version', 'N/A')}")
print(f"Number of contracts (articles): {len(cuad_raw['data'])}")


CUAD version: aok_v1.0
Number of contracts (articles): 510


In [ ]:
# Convert SQuAD-format JSON into a flat list of examples
def squad_json_to_examples(raw_data):
    """Flatten SQuAD-format JSON into a list of dicts for HuggingFace Dataset."""
    examples = []
    for article in raw_data["data"]:
        title = article.get("title", "")
        for paragraph in article["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                example = {
                    "id": qa["id"],
                    "title": title,
                    "context": context,
                    "question": qa["question"],
                    "answers": {
                        "text": [a["text"] for a in qa.get("answers", [])],
                        "answer_start": [a["answer_start"] for a in qa.get("answers", [])],
                    },
                    "is_impossible": qa.get("is_impossible", False),
                }
                examples.append(example)
    return examples

all_examples = squad_json_to_examples(cuad_raw)
print(f"Total examples: {len(all_examples)}")

# Create a HuggingFace Dataset from the list
full_dataset = Dataset.from_list(all_examples)
print(full_dataset)

Total examples: 20910
Dataset({
    features: ['id', 'title', 'context', 'question', 'answers', 'is_impossible'],
    num_rows: 20910
})


In [ ]:
# Explore one example
example = full_dataset[0]
print("ID:", example["id"])
print("Question:", example["question"])
print("Context (first 500 chars):", example["context"][:500])
print("Answers:", example["answers"])

ID: LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Document Name
Question: Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract
Context (first 500 chars): EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.

                                    RECITALS

         A. The  Company's  Business.  The Company is  presently  engaged in the business  of selling an energy  efficiency  device,  which is  referred to as an 
Answers: {'answer_start': [44], 'text': ['DISTRIBUTOR AGREEMENT']}


In [ ]:
# Show the distribution of clause types (questions)
from collections import Counter

question_counts = Counter(ex["question"] for ex in full_dataset)
print(f"Number of unique clause types: {len(question_counts)}")
print("\nClause type distribution (top 10):")
for q, c in question_counts.most_common(10):
    print(f"  [{c:5d}] {q[:80]}")

Number of unique clause types: 41

Clause type distribution (top 10):
  [  510] Highlight the parts (if any) of this contract related to "Document Name" that sh
  [  510] Highlight the parts (if any) of this contract related to "Parties" that should b
  [  510] Highlight the parts (if any) of this contract related to "Agreement Date" that s
  [  510] Highlight the parts (if any) of this contract related to "Effective Date" that s
  [  510] Highlight the parts (if any) of this contract related to "Expiration Date" that 
  [  510] Highlight the parts (if any) of this contract related to "Renewal Term" that sho
  [  510] Highlight the parts (if any) of this contract related to "Notice Period To Termi
  [  510] Highlight the parts (if any) of this contract related to "Governing Law" that sh
  [  510] Highlight the parts (if any) of this contract related to "Most Favored Nation" t
  [  510] Highlight the parts (if any) of this contract related to "Non-Compete" that shou


In [ ]:
# Count how many examples have an answer vs. are "impossible" (no answer)
has_answer = sum(1 for ex in full_dataset if len(ex["answers"]["text"]) > 0)
no_answer  = len(full_dataset) - has_answer
print(f"Has answer:  {has_answer}")
print(f"No answer:   {no_answer}")
print(f"Answerable:  {has_answer / len(full_dataset) * 100:.1f}%")

Has answer:  6702
No answer:   14208
Answerable:  32.1%


In [ ]:
# First split: 90% train+val, 10% test
split1 = full_dataset.train_test_split(test_size=0.1, seed=42)
# Second split: from the 90%, take ~11% as validation (= ~10% of original)
split2 = split1["train"].train_test_split(test_size=0.111, seed=42)

train_dataset = split2["train"]
val_dataset   = split2["test"]
test_dataset  = split1["test"]

print(f"Train:      {len(train_dataset)}")
print(f"Validation: {len(val_dataset)}")
print(f"Test:       {len(test_dataset)}")

Train:      16730
Validation: 2089
Test:       2091


In [ ]:
model_name = "nlpaueb/legal-bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)
model.to(device)

print(f"Model parameters: {model.num_parameters():,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

BertForQuestionAnswering LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored whe

Model parameters: 108,893,186


In [ ]:
from transformers import AutoTokenizer

max_length = 384
doc_stride = 128
model_name = "nlpaueb/legal-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
pad_on_right = tokenizer.padding_side == "right"


def prepare_train_features(examples):
    """Tokenize and align answer spans for training."""
    questions = [q.strip() for q in examples["question"]]
    contexts  = examples["context"]

    tokenized = tokenizer(
        questions if pad_on_right else contexts,
        contexts  if pad_on_right else questions,
        truncation="only_second" if pad_on_right else "only_first",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    tokenized["start_positions"] = []
    tokenized["end_positions"]   = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = tokenized.sequence_ids(i)

        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]

        # If no answer -> point to CLS
        if len(answers["answer_start"]) == 0 or answers["answer_start"][0] == -1:
            tokenized["start_positions"].append(cls_index)
            tokenized["end_positions"].append(cls_index)
            continue

        start_char = answers["answer_start"][0]
        end_char   = start_char + len(answers["text"][0])

        # Find the context token boundary
        ctx_id = 1 if pad_on_right else 0
        token_start = 0
        while token_start < len(sequence_ids) and sequence_ids[token_start] != ctx_id:
            token_start += 1
        token_end = len(input_ids) - 1
        while token_end >= 0 and sequence_ids[token_end] != ctx_id:
            token_end -= 1

        # If the answer is out of the span, label as CLS
        if token_start >= len(offsets) or token_end < 0:
            tokenized["start_positions"].append(cls_index)
            tokenized["end_positions"].append(cls_index)
            continue

        if offsets[token_start][0] > start_char or offsets[token_end][1] < end_char:
            tokenized["start_positions"].append(cls_index)
            tokenized["end_positions"].append(cls_index)
        else:
            # Move to the token that contains start_char
            while token_start < len(offsets) and offsets[token_start][0] <= start_char:
                token_start += 1
            tokenized["start_positions"].append(token_start - 1)
            # Move to the token that contains end_char
            while token_end >= 0 and offsets[token_end][1] >= end_char:
                token_end -= 1
            tokenized["end_positions"].append(token_end + 1)

    return tokenized


print("Tokenizing training set...")
tokenized_train = train_dataset.map(
    prepare_train_features,
    batched=True,
    remove_columns=train_dataset.column_names,
)
print(f"Training features: {len(tokenized_train)}")

Tokenizing training set...


Map:   0%|          | 0/16730 [00:00<?, ? examples/s]

In [ ]:
def prepare_validation_features(examples):
    """Tokenize for validation/test — keep offset_mapping and example_id for post-processing."""
    questions = [q.strip() for q in examples["question"]]
    contexts  = examples["context"]

    tokenized = tokenizer(
        questions if pad_on_right else contexts,
        contexts  if pad_on_right else questions,
        truncation="only_second" if pad_on_right else "only_first",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")

    tokenized["example_id"] = []
    for i in range(len(tokenized["input_ids"])):
        sample_index = sample_mapping[i]
        tokenized["example_id"].append(examples["id"][sample_index])

        # Set offsets of non-context tokens to None so they are ignored
        sequence_ids = tokenized.sequence_ids(i)
        ctx_id = 1 if pad_on_right else 0
        tokenized["offset_mapping"][i] = [
            (o if sequence_ids[k] == ctx_id else None)
            for k, o in enumerate(tokenized["offset_mapping"][i])
        ]

    return tokenized


print("Tokenizing validation set...")
tokenized_val = val_dataset.map(
    prepare_validation_features,
    batched=True,
    remove_columns=val_dataset.column_names,
)
print(f"Validation features: {len(tokenized_val)}")

print("Tokenizing test set...")
tokenized_test = test_dataset.map(
    prepare_validation_features,
    batched=True,
    remove_columns=test_dataset.column_names,
)
print(f"Test features: {len(tokenized_test)}")

In [ ]:
training_args = TrainingArguments(
    output_dir="./legal-bert-cuad",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,      # T4 can handle 16 with fp16
    per_device_eval_batch_size=32,       # eval uses less memory
    gradient_accumulation_steps=2,       # effective batch size = 32
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=True,                           # T4 has great fp16 support
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=100,
    report_to="none",
)

# The validation set for the Trainer uses the TRAIN-style tokenization
# (with start_positions/end_positions) so that eval_loss is computed.
tokenized_val_for_trainer = val_dataset.map(
    prepare_train_features,
    batched=True,
    remove_columns=val_dataset.column_names,
)



In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val_for_trainer,
    processing_class=tokenizer,
    data_collator=default_data_collator,
)

print("Starting training...")
trainer.train()

In [ ]:
# Save the fine-tuned model
trainer.save_model("./legal-bert-cuad-final")
tokenizer.save_pretrained("./legal-bert-cuad-final")
print("Model saved to ./legal-bert-cuad-final")

In [ ]:
def postprocess_qa_predictions(
    examples, features, raw_predictions, n_best_size=20, max_answer_length=128
):
    """
    Post-process raw start/end logits into text predictions.
    Returns a dict mapping example id -> predicted answer string.
    """
    all_start_logits, all_end_logits = raw_predictions

    # Map each example to its feature indices
    example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
    features_per_example = collections.defaultdict(list)
    for i, feat_id in enumerate(features["example_id"]):
        features_per_example[example_id_to_index[feat_id]].append(i)

    predictions = collections.OrderedDict()

    for example_index, example in enumerate(examples):
        feature_indices = features_per_example[example_index]
        context = example["context"]

        min_null_score = None
        valid_answers = []

        for feature_index in feature_indices:
            start_logits = all_start_logits[feature_index]
            end_logits   = all_end_logits[feature_index]
            offset_mapping = features["offset_mapping"][feature_index]

            cls_score = start_logits[0] + end_logits[0]
            if min_null_score is None or cls_score < min_null_score:
                min_null_score = cls_score

            start_indexes = np.argsort(start_logits)[-1: -n_best_size - 1: -1].tolist()
            end_indexes   = np.argsort(end_logits)[-1: -n_best_size - 1: -1].tolist()

            for start_index in start_indexes:
                for end_index in end_indexes:
                    if (
                        start_index >= len(offset_mapping)
                        or end_index >= len(offset_mapping)
                        or offset_mapping[start_index] is None
                        or offset_mapping[end_index] is None
                        or end_index < start_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue
                    start_char = offset_mapping[start_index][0]
                    end_char   = offset_mapping[end_index][1]
                    valid_answers.append(
                        {
                            "score": start_logits[start_index] + end_logits[end_index],
                            "text": context[start_char:end_char],
                        }
                    )

        if valid_answers:
            best_answer = sorted(valid_answers, key=lambda x: x["score"], reverse=True)[0]
        else:
            best_answer = {"text": "", "score": 0.0}

        # Use null score threshold: predict empty if null score is higher
        if min_null_score is not None and min_null_score > best_answer["score"]:
            predictions[example["id"]] = ""
        else:
            predictions[example["id"]] = best_answer["text"]

    return predictions

In [ ]:
# Run predictions on the validation set
# We need features WITHOUT start/end positions but WITH offset_mapping & example_id
val_features_for_pred = tokenized_val.remove_columns(
    [c for c in tokenized_val.column_names if c not in
     ["input_ids", "attention_mask", "token_type_ids", "offset_mapping", "example_id"]]
)

# The Trainer.predict needs only model inputs
val_features_model = tokenized_val.remove_columns(
    ["offset_mapping", "example_id"]
)

raw_predictions = trainer.predict(val_features_model)
print(f"Prediction output keys: start_logits shape={raw_predictions.predictions[0].shape}, "
      f"end_logits shape={raw_predictions.predictions[1].shape}")

In [ ]:
# Post-process predictions
final_predictions = postprocess_qa_predictions(
    val_dataset, val_features_for_pred, raw_predictions.predictions
)

# Compute clause-level metrics
# A positive = the model predicts a non-empty answer
# Ground truth positive = the example has a non-empty answer
y_true = []
y_pred = []
for example in val_dataset:
    has_gt = len(example["answers"]["text"]) > 0 and len(example["answers"]["text"][0]) > 0
    has_pred = len(final_predictions.get(example["id"], "")) > 0
    y_true.append(1 if has_gt else 0)
    y_pred.append(1 if has_pred else 0)

precision = precision_score(y_true, y_pred, zero_division=0)
recall    = recall_score(y_true, y_pred, zero_division=0)
f1        = f1_score(y_true, y_pred, zero_division=0)

print("=" * 50)
print("Clause-Level Detection Metrics (Validation Set)")
print("=" * 50)
print(f"Precision: {precision:.4f}  (target >= 0.88)")
print(f"Recall:    {recall:.4f}  (target >= 0.88)")
print(f"F1-Score:  {f1:.4f}")
print("=" * 50)

In [ ]:
# Show some example predictions
print("\nSample Predictions vs Ground Truth:\n")
for i, example in enumerate(val_dataset.select(range(min(10, len(val_dataset))))):
    gt_texts = example["answers"]["text"]
    gt = gt_texts[0] if gt_texts else "<no clause>"
    pred = final_predictions.get(example["id"], "")
    pred = pred if pred else "<no clause>"
    print(f"[{i+1}] Question: {example['question'][:70]}...")
    print(f"    Ground Truth: {gt[:100]}")
    print(f"    Prediction:   {pred[:100]}")
    print()

In [ ]:
from transformers import pipeline

# Load fine-tuned model for inference
qa_pipeline = pipeline(
    "question-answering",
    model="./legal-bert-cuad-final",
    tokenizer="./legal-bert-cuad-final",
    device=0,    # force GPU 0 (your T4)
)

# Example contract text
contract_text = (
    "1. Indemnification. Developer shall indemnify, defend and hold harmless "
    "Client from and against any and all claims, demands, liabilities, costs, "
    "and expenses arising from Developer's negligence. Client holds no liability "
    "over damages caused by Client's misuse. "
    "2. Non-Compete. Developer agrees that, for a period of two (2) years after "
    "termination, Developer shall not engage in any business competitive with "
    "Client's business within the United States. "
    "3. Termination. Either party may terminate this agreement with 30 days' "
    "written notice. Upon termination, all intellectual property rights revert "
    "to Client."
)

# CUAD clause-type queries (these mirror the actual CUAD question phrasing)
clause_queries = [
    "Highlight the parts (if any) of this contract regarding Indemnification.",
    "Highlight the parts (if any) of this contract regarding Non-Compete.",
    "Highlight the parts (if any) of this contract regarding Termination For Convenience.",
    "Highlight the parts (if any) of this contract regarding Intellectual Property Ownership Assignment.",
    "Highlight the parts (if any) of this contract regarding Liability.",
]

print("=" * 60)
print("Clause Extraction Results")
print("=" * 60)
for query in clause_queries:
    result = qa_pipeline(question=query, context=contract_text)
    clause_type = query.split("regarding ")[-1].rstrip(".")
    print(f"\n[{clause_type}]")
    print(f"  Answer:     {result['answer']}")
    print(f"  Confidence: {result['score']:.4f}")
    print(f"  Span:       ({result['start']}, {result['end']})")

In [ ]:
# Map clause types to risk levels
RISK_LEVELS = {
    "Indemnification": "HIGH",
    "Non-Compete": "HIGH",
    "Termination For Convenience": "MEDIUM",
    "Intellectual Property Ownership Assignment": "HIGH",
    "Liability": "HIGH",
    "Uncapped Liability": "CRITICAL",
    "Cap On Liability": "MEDIUM",
    "Liquidated Damages": "HIGH",
    "Non-Solicitation": "MEDIUM",
    "Anti-Assignment": "LOW",
    "Revenue/Profit Sharing": "MEDIUM",
    "Audit Rights": "LOW",
    "Insurance": "LOW",
}

CONFIDENCE_THRESHOLD = 0.1

def score_contract(contract, queries_with_types, pipe):
    """Run all clause-type queries and return risk report."""
    findings = []
    for query, clause_type in queries_with_types:
        result = pipe(question=query, context=contract)
        if result["score"] >= CONFIDENCE_THRESHOLD and result["answer"].strip():
            risk = RISK_LEVELS.get(clause_type, "UNKNOWN")
            findings.append({
                "clause_type": clause_type,
                "risk_level": risk,
                "extracted_text": result["answer"],
                "confidence": result["score"],
                "span": (result["start"], result["end"]),
            })
    return findings

queries_with_types = [
    ("Highlight the parts (if any) of this contract regarding Indemnification.", "Indemnification"),
    ("Highlight the parts (if any) of this contract regarding Non-Compete.", "Non-Compete"),
    ("Highlight the parts (if any) of this contract regarding Termination For Convenience.", "Termination For Convenience"),
    ("Highlight the parts (if any) of this contract regarding Intellectual Property Ownership Assignment.", "Intellectual Property Ownership Assignment"),
    ("Highlight the parts (if any) of this contract regarding Liability.", "Liability"),
]

findings = score_contract(contract_text, queries_with_types, qa_pipeline)

print("\n" + "=" * 60)
print("RISK REPORT")
print("=" * 60)
if not findings:
    print("No risky clauses detected.")
else:
    for f in findings:
        risk_icon = {"CRITICAL": "🔴", "HIGH": "🟠", "MEDIUM": "🟡", "LOW": "🟢"}.get(f["risk_level"], "⚪")
        print(f"\n{risk_icon} [{f['risk_level']}] {f['clause_type']}")
        print(f"  Confidence: {f['confidence']:.4f}")
        print(f"  Text: \"{f['extracted_text'][:120]}\"")
    print(f"\nTotal findings: {len(findings)}")

# Version 2

In [ ]:
%pip install -q transformers datasets evaluate accelerate scikit-learn captum torch

In [ ]:
import torch
import gc
import numpy as np
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from datasets import load_dataset
from captum.attr import LayerIntegratedGradients

In [ ]:
# ==========================================
# 1. HARDWARE ENFORCEMENT (Strict GPU Usage)
# ==========================================
# Ensure we are using the T4 GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != 'cuda':
    raise SystemError("GPU not detected! Go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU")
print(f"Using device: {device}")

# Clear any lingering VRAM
torch.cuda.empty_cache()
gc.collect()

Using device: cuda


132

In [ ]:
# ==========================================
# 2. MODEL & TOKENIZER INITIALIZATION
# ==========================================
model_name = "nlpaueb/legal-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 3 classes based on typical fairness datasets (0: Fair, 1: Potentially Unfair, 2: Unfair)
# Loading directly to GPU to save System RAM
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were new

In [ ]:
# ==========================================
# 3. DATA LOADING & PREPROCESSING (FIXED)
# ==========================================
print("Loading dataset...")
dataset = load_dataset("CodeHima/TOS_Dataset")

# Map the string labels from the dataset to integer values
label_map = {
    "clearly_fair": 0,
    "potentially_unfair": 1,
    "clearly_unfair": 2
}

def tokenize_function(examples):
    # 1. Tokenize the correct column ('sentence' instead of 'text')
    tokenized = tokenizer(
        examples["sentence"],
        truncation=True,
        max_length=256
    )

    # 2. Create the exact 'labels' column the Trainer requires
    tokenized["labels"] = [label_map[label] for label in examples["unfairness_level"]]

    return tokenized

print("Tokenizing and mapping dataset...")
# Map the function and strip out the original string columns
# If we leave strings in the dataset, the PyTorch data collator will throw an error
tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Loading dataset...
Tokenizing and mapping dataset...


Map:   0%|          | 0/5378 [00:00<?, ? examples/s]

Map:   0%|          | 0/415 [00:00<?, ? examples/s]

Map:   0%|          | 0/1038 [00:00<?, ? examples/s]

In [ ]:
# ==========================================
# 4. METRICS (Targeting 0.88+ Precision/Recall)
# ==========================================
clf_metrics = evaluate.combine(["precision", "recall", "f1", "accuracy"])

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return clf_metrics.compute(predictions=predictions, references=labels, average="weighted")

In [ ]:
print(dataset['train'].features)

{'sentence': Value('string'), 'unfairness_level': Value('string'), '__index_level_0__': Value('int64')}


In [ ]:
# ==========================================
# 4. METRICS (FIXED)
# ==========================================
import evaluate
import numpy as np

# Load metrics individually instead of using combine()
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    # Accuracy does NOT take an 'average' argument
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]

    # Precision, Recall, and F1 DO require the 'average' argument for our 3 classes
    precision = precision_metric.compute(predictions=predictions, references=labels, average="weighted")["precision"]
    recall = recall_metric.compute(predictions=predictions, references=labels, average="weighted")["recall"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"]

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
# ==========================================
# 5. TRAINING ARGUMENTS (T4 Optimized)
# ==========================================
training_args = TrainingArguments(
    output_dir="./legal_bert_tos_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16, # Optimized for T4 16GB VRAM
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True, # MUST be True for T4: uses mixed precision, halving VRAM and doubling speed
    push_to_hub=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
# ==========================================
# 6. TRAIN & EVALUATE
# ==========================================
print("Starting Training on GPU...")
trainer.train()

print("Evaluating Model against 0.88+ target...")
results = trainer.evaluate()
print(f"Evaluation Results: {results}")

Starting Training on GPU...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.568972,0.838150,0.841425,0.838150,0.817211
2,0.271546,0.485457,0.851638,0.846550,0.851638,0.846108
3,0.192209,0.595450,0.842967,0.842569,0.842967,0.841190


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Evaluating Model against 0.88+ target...


Evaluation Results: {'eval_loss': 0.4861046075820923, 'eval_accuracy': 0.8516377649325626, 'eval_precision': 0.8465498350808864, 'eval_recall': 0.8516377649325626, 'eval_f1': 0.8461076047898382, 'eval_runtime': 2.6189, 'eval_samples_per_second': 396.344, 'eval_steps_per_second': 24.819, 'epoch': 3.0}


In [ ]:
# ==========================================
# 7. EXPLAINABILITY DEMO (Integrated Gradients)
# ==========================================
print("\n--- Testing with Real-Life Example ---")

# Real-life predatory auto-renewal/liability shift example
sample_text = "The user agrees to automatically renew this subscription annually at the then-current standard rate. Furthermore, the provider assumes absolutely no liability for any data loss, intellectual property theft, or damages incurred while using our service."

# Move inputs strictly to GPU
inputs = tokenizer(sample_text, return_tensors="pt").to(device)

# Inference
model.eval()
with torch.no_grad():
    outputs = model(**inputs)
    prediction = torch.argmax(outputs.logits, dim=-1).item()

labels_map = {0: "Fair", 1: "Potentially Unfair", 2: "Highly Predatory / Unfair"}
print(f"Clause: '{sample_text}'")
print(f"Risk Assessment: **{labels_map.get(prediction, 'Unknown')}**")

# --- Integrated Gradients setup using Captum ---
# We need a wrapper function for Captum that extracts the logits
def predict_forward(inputs, attention_mask=None):
    return model(inputs, attention_mask=attention_mask)[0]

# Initialize Layer Integrated Gradients targeting the Word Embeddings of Legal-BERT
lig = LayerIntegratedGradients(predict_forward, model.bert.embeddings.word_embeddings)

# Generate baseline (zeros) for comparison
input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]
baseline = torch.zeros_like(input_ids).to(device)

# Calculate attributions on the GPU
attributions, delta = lig.attribute(
    inputs=input_ids,
    baselines=baseline,
    additional_forward_args=(attention_mask,),
    target=prediction,
    return_convergence_delta=True
)

# Summarize attributions to get a single score per token
attributions_sum = attributions.sum(dim=-1).squeeze(0).cpu().detach().numpy()
tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze(0).cpu().tolist())

# Display the highest risk words
print("\n--- Integrated Gradients: Risk Attribution Heatmap Highlights ---")
token_scores = list(zip(tokens, attributions_sum))
# Filter out special tokens and sort by highest attribution
filtered_scores = [(t, s) for t, s in token_scores if t not in ['[CLS]', '[SEP]', '[PAD]']]
filtered_scores.sort(key=lambda x: x[1], reverse=True)

print("Words contributing most to the 'Unfair' classification:")
for token, score in filtered_scores[:8]:
    # Clean up subwords (##) for cleaner output
    clean_token = token.replace("##", "")
    print(f" - {clean_token}: {score:.4f}")

# Clean up memory at the end
del model
torch.cuda.empty_cache()


--- Testing with Real-Life Example ---
Clause: 'The user agrees to automatically renew this subscription annually at the then-current standard rate. Furthermore, the provider assumes absolutely no liability for any data loss, intellectual property theft, or damages incurred while using our service.'
Risk Assessment: **Potentially Unfair**

--- Integrated Gradients: Risk Attribution Heatmap Highlights ---
Words contributing most to the 'Unfair' classification:
 - liability: 0.3915
 - annually: 0.2426
 - assumes: 0.2311
 - absolutely: 0.2199
 - to: 0.1665
 - at: 0.1491
 - the: 0.1485
 - ,: 0.1449


# Version 3

In [ ]:
import os
import json
import torch
import gc
import evaluate
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [ ]:
# ==========================================
# 1. HARDWARE SETUP (Strict GPU)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != 'cuda':
    raise SystemError("GPU not detected! Enable T4 GPU in Colab Runtime.")

torch.cuda.empty_cache()
gc.collect()

153

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ==========================================
# 2. CUAD JSON PARSING & DATASET CREATION
# ==========================================
print("Parsing CUAD JSON...")

cuad_path = os.path.join("drive/MyDrive/Sem 2/AIG230/Project/Dataset", "CUAD_v1.json")

with open(cuad_path, "r", encoding="utf-8") as f:
    cuad_raw = json.load(f)

records = []
label_names = []

# Extract clauses and their categories from the SQuAD-formatted JSON
for doc in cuad_raw["data"]:
    for para in doc["paragraphs"]:
        for qa in para["qas"]:
            if not qa["is_impossible"]: # We only want actual extracted clauses
                question = qa["title"] if "title" in qa else qa["question"]
                # Clean up the label name
                category = question.split("related to: ")[-1].strip() if "related to: " in question else question

                if category not in label_names:
                    label_names.append(category)

                label_id = label_names.index(category)

                for ans in qa["answers"]:
                    text = ans["text"].strip()
                    if len(text) > 15: # Filter out single-word junk
                        records.append({"text": text, "label": label_id})

df = pd.DataFrame(records)
num_classes = len(label_names)
print(f"Extracted {len(df)} clauses across {num_classes} unique categories.")

# Convert to Hugging Face Dataset and split into Train/Test
full_dataset = Dataset.from_pandas(df)
train_test_split = full_dataset.train_test_split(test_size=0.15, seed=42)
dataset = DatasetDict({
    'train': train_test_split['train'],
    'test': train_test_split['test']
})

Parsing CUAD JSON...
Extracted 12254 clauses across 41 unique categories.


In [ ]:
# ==========================================
# 3. INITIALIZE MODEL & TOKENIZER
# ==========================================
model_name = "nlpaueb/legal-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_classes,
    id2label={i: label for i, label in enumerate(label_names)},
    label2id={label: i for i, label in enumerate(label_names)}
).to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were new

In [ ]:
# ==========================================
# 4. TOKENIZATION
# ==========================================
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256 # Cap length to save GPU VRAM
    )

print("Tokenizing data...")
tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Tokenizing data...


Map:   0%|          | 0/10415 [00:00<?, ? examples/s]

Map:   0%|          | 0/1839 [00:00<?, ? examples/s]

In [ ]:
# ==========================================
# 5. METRICS & TRAINING
# ==========================================
acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    acc = acc_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "f1": f1}

save_directory = "./saved_cuad_legal_bert"

training_args = TrainingArguments(
    output_dir=save_directory,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True, # Critical for T4 GPU performance
    push_to_hub=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    # tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Starting GPU Training...")
trainer.train()

Starting GPU Training...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.863209,0.861606,0.772703,0.735326
2,0.871040,0.690257,0.794997,0.776109
3,0.695103,0.663833,0.790647,0.772418


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1953, training_loss=1.017679178342414, metrics={'train_runtime': 514.2852, 'train_samples_per_second': 60.754, 'train_steps_per_second': 3.798, 'total_flos': 2915311504857204.0, 'train_loss': 1.017679178342414, 'epoch': 3.0})

In [ ]:
# ==========================================
# 6. EXPLICITLY SAVE THE MODEL AND TOKENIZER
# ==========================================
print(f"Saving finalized model and tokenizer to {save_directory}...")
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)

# Clear VRAM so we can demonstrate loading
del model
del trainer
torch.cuda.empty_cache()
gc.collect()

Saving finalized model and tokenizer to ./saved_cuad_legal_bert...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

4936

In [ ]:
# ==========================================
# 7. LOAD AND REUSE THE SAVED MODEL
# ==========================================
print("\n--- Testing Reloaded Model ---")

# Load directly from the local directory we just saved to
loaded_tokenizer = AutoTokenizer.from_pretrained(save_directory)
loaded_model = AutoModelForSequenceClassification.from_pretrained(save_directory).to(device)
loaded_model.eval()

# A sample clause to test
sample_text = "The Receiving Party agrees not to use any Confidential Information of the Disclosing Party for any purpose except to evaluate and engage in discussions concerning a potential business relationship between the parties."

# Run inference
inputs = loaded_tokenizer(sample_text, return_tensors="pt", truncation=True, max_length=256).to(device)

with torch.no_grad():
    outputs = loaded_model(**inputs)
    prediction_id = torch.argmax(outputs.logits, dim=-1).item()

# Use the model's built-in id2label mapping (which we configured during initialization)
predicted_label = loaded_model.config.id2label[prediction_id]

print(f"Text: '{sample_text}'")
print(f"Predicted Clause Type: **{predicted_label}**")


--- Testing Reloaded Model ---


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Text: 'The Receiving Party agrees not to use any Confidential Information of the Disclosing Party for any purpose except to evaluate and engage in discussions concerning a potential business relationship between the parties.'
Predicted Clause Type: **Highlight the parts (if any) of this contract related to "Non-Compete" that should be reviewed by a lawyer. Details: Is there a restriction on the ability of a party to compete with the counterparty or operate in a certain geography or business or technology sector? **


In [ ]:
import shutil
source = '/content/saved_cuad_legal_bert'

destination = '/content/drive/MyDrive/Sem 2/AIG230/Project/Saved Model V3'

shutil.move(source, destination)

'/content/drive/MyDrive/Sem 2/AIG230/Project/Saved Model V3/saved_cuad_legal_bert'

# Version 4

In [ ]:
import os
import json
import torch
import gc
import evaluate
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [ ]:
# ==========================================
# 1. HARDWARE SETUP (Strict GPU)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != 'cuda':
    raise SystemError("GPU not detected! Enable T4 GPU in Colab Runtime.")

torch.cuda.empty_cache()
gc.collect()

102

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ==========================================
# 2. CUAD JSON PARSING & DATASET CREATION
# ==========================================
print("Parsing CUAD JSON...")

cuad_path = os.path.join("drive/MyDrive/Sem 2/AIG230/Project/Dataset", "CUAD_v1.json")

with open(cuad_path, "r", encoding="utf-8") as f:
    cuad_raw = json.load(f)

records = []
label_names = []

# Extract clauses and their categories from the SQuAD-formatted JSON
for doc in cuad_raw["data"]:
    for para in doc["paragraphs"]:
        for qa in para["qas"]:
            if not qa["is_impossible"]: # We only want actual extracted clauses
                question = qa["title"] if "title" in qa else qa["question"]
                # Clean up the label name
                category = question.split("related to: ")[-1].strip() if "related to: " in question else question

                if category not in label_names:
                    label_names.append(category)

                label_id = label_names.index(category)

                for ans in qa["answers"]:
                    text = ans["text"].strip()
                    if len(text) > 15: # Filter out single-word junk
                        records.append({"text": text, "label": label_id})

df = pd.DataFrame(records)
num_classes = len(label_names)
print(f"Extracted {len(df)} clauses across {num_classes} unique categories.")

# Convert to Hugging Face Dataset and split into Train/Test
full_dataset = Dataset.from_pandas(df)
train_test_split = full_dataset.train_test_split(test_size=0.15, seed=42)
dataset = DatasetDict({
    'train': train_test_split['train'],
    'test': train_test_split['test']
})

Parsing CUAD JSON...
Extracted 12254 clauses across 41 unique categories.


In [ ]:
# ==========================================
# 3. INITIALIZE MODEL & TOKENIZER
# ==========================================
model_name = "nlpaueb/legal-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_classes,
    id2label={i: label for i, label in enumerate(label_names)},
    label2id={label: i for i, label in enumerate(label_names)}
).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were new

In [ ]:
# ==========================================
# 4. TOKENIZATION
# ==========================================
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=320 # Slightly longer context can improve classification quality
    )

print("Tokenizing data...")
tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Tokenizing data...


Map:   0%|          | 0/10415 [00:00<?, ? examples/s]

Map:   0%|          | 0/1839 [00:00<?, ? examples/s]

In [ ]:
# ==========================================
# 5. METRICS & TRAINING (TUNED)
# ==========================================
acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    acc = acc_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "f1": f1}

save_directory = "./saved_cuad_legal_bert"

training_args = TrainingArguments(
    output_dir=save_directory,
    learning_rate=1.5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=6,
    weight_decay=0.02,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    metric_for_best_model="f1",
    greater_is_better=True,
    load_best_model_at_end=True,
    fp16=True, # Critical for T4 GPU performance
    push_to_hub=False,
    report_to="none"
 )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
 )

print("Starting GPU Training...")
trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting GPU Training...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,5.961120,1.106841,0.723219,0.668570
2,2.228461,0.736290,0.789560,0.762082
3,1.561264,0.675755,0.794997,0.777750
4,1.059869,0.644183,0.794997,0.785381
5,0.949516,0.644772,0.789560,0.781618
6,0.862663,0.643617,0.787928,0.781050


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=3906, training_loss=1.8676063627447157, metrics={'train_runtime': 748.3017, 'train_samples_per_second': 83.509, 'train_steps_per_second': 5.22, 'total_flos': 4925566976505684.0, 'train_loss': 1.8676063627447157, 'epoch': 6.0})

In [ ]:
# ==========================================
# 6. EXPLICITLY SAVE THE MODEL AND TOKENIZER
# ==========================================
print(f"Saving finalized model and tokenizer to {save_directory}...")
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)

# Clear VRAM so we can demonstrate loading
del model
del trainer
torch.cuda.empty_cache()
gc.collect()

Saving finalized model and tokenizer to ./saved_cuad_legal_bert...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

4936

In [ ]:
# ==========================================
# 7. LOAD AND REUSE THE SAVED MODEL
# ==========================================
print("\n--- Testing Reloaded Model ---")

# Load directly from the local directory we just saved to
loaded_tokenizer = AutoTokenizer.from_pretrained(save_directory)
loaded_model = AutoModelForSequenceClassification.from_pretrained(save_directory).to(device)
loaded_model.eval()

# A sample clause to test
sample_text = "The Receiving Party agrees not to use any Confidential Information of the Disclosing Party for any purpose except to evaluate and engage in discussions concerning a potential business relationship between the parties."

# Run inference
inputs = loaded_tokenizer(sample_text, return_tensors="pt", truncation=True, max_length=256).to(device)

with torch.no_grad():
    outputs = loaded_model(**inputs)
    prediction_id = torch.argmax(outputs.logits, dim=-1).item()

# Use the model's built-in id2label mapping (which we configured during initialization)
predicted_label = loaded_model.config.id2label[prediction_id]

print(f"Text: '{sample_text}'")
print(f"Predicted Clause Type: **{predicted_label}**")


--- Testing Reloaded Model ---


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Text: 'The Receiving Party agrees not to use any Confidential Information of the Disclosing Party for any purpose except to evaluate and engage in discussions concerning a potential business relationship between the parties.'
Predicted Clause Type: **Highlight the parts (if any) of this contract related to "Non-Compete" that should be reviewed by a lawyer. Details: Is there a restriction on the ability of a party to compete with the counterparty or operate in a certain geography or business or technology sector? **


In [ ]:
import shutil
source = '/content/saved_cuad_legal_bert'

destination = '/content/drive/MyDrive/Sem 2/AIG230/Project/Saved Model V3'

shutil.move(source, destination)

'/content/drive/MyDrive/Sem 2/AIG230/Project/Saved Model V3/saved_cuad_legal_bert'